# Variables

# Create a log successful and failed subtypefolders

In [1]:
import os
import shutil
import re

# Folder setup
log_folder = "/gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs"
run_log_path = os.path.join(log_folder, "run_log.txt")
success_folder = os.path.join(log_folder, "successful")
fail_folder = os.path.join(log_folder, "failed")

os.makedirs(success_folder, exist_ok=True)
os.makedirs(fail_folder, exist_ok=True)

# Parse run_log.txt
success_hashes = []
fail_hashes = []

with open(run_log_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    match = re.match(r"==== Processing (.+\.mod) ====", line)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        next_line = lines[i + 1] if i + 1 < len(lines) else ""
        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)

# Move .log files based on success/failure
def move_log_files(hashes, destination):
    for hash_id in hashes:
        log_file = f"{hash_id}.log"
        src = os.path.join(log_folder, log_file)
        dst = os.path.join(destination, log_file)
        if os.path.exists(src):
            shutil.copy2(src, dst)

move_log_files(success_hashes, success_folder)
move_log_files(fail_hashes, fail_folder)

print(f"✅ Moved {len(success_hashes)} successful logs to {success_folder}")
print(f"❌ Moved {len(fail_hashes)} failed logs to {fail_folder}")


✅ Moved 426 successful logs to /gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs/successful
❌ Moved 128 failed logs to /gpfs/gibbs/project/mcdougal/imc33/mod-extract/logs/failed


In [3]:
log_files = set(f.replace(".log", "") for f in os.listdir(log_folder) if f.endswith(".log"))
mentioned_hashes = set(success_hashes + fail_hashes)
unmatched_logs = log_files - mentioned_hashes
print("Not matched in run_log.txt:", unmatched_logs)

Not matched in run_log.txt: set()


In [4]:
# Check for mod files mentioned in run_log.txt but not labeled as success or failure
processed_hashes = set()
for i, line in enumerate(lines):
    match = re.match(r"==== Processing (.+\.mod) ====", line)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        processed_hashes.add(hash_id)

        # Check next line for status
        next_line = lines[i + 1] if i + 1 < len(lines) else ""
        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)

# Now find any processed hashes that were not categorized
uncategorized = processed_hashes - set(success_hashes) - set(fail_hashes)
print("✅ Total processed:", len(processed_hashes))
print("✅ Total successes:", len(success_hashes))
print("❌ Total failures:", len(fail_hashes))
print("🤷 Not classified:", len(uncategorized))
print("List of not classified:", uncategorized)


✅ Total processed: 554
✅ Total successes: 852
❌ Total failures: 256
🤷 Not classified: 0
List of not classified: set()


In [5]:
for h in success_hashes:
    if h not in processed_hashes:
        print("Unexpected success hash not in processed set:", h)


In [6]:
success_hashes = []
fail_hashes = []
processed_hashes = []

for i in range(len(lines) - 1):
    current = lines[i]
    next_line = lines[i + 1]

    match = re.match(r"==== Processing (.+\.mod) ====", current)
    if match:
        mod_filename = match.group(1)
        hash_id = mod_filename.replace(".mod", "")
        processed_hashes.append(hash_id)

        if "Successfully processed" in next_line:
            success_hashes.append(hash_id)
        elif "failed" in next_line.lower():
            fail_hashes.append(hash_id)


## Copy failed nmodl files to data folder

In [9]:
import os
import shutil

# Get absolute path to the project root (go up one level from 'code/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Absolute paths
failed_log_dir = os.path.join(project_root, "logs", "failed")
mod_src_dir = os.path.join(project_root, "data", "raw", "nmodl")
mod_dst_dir = os.path.join(project_root, "data", "raw", "nmodl-failed")

# Create the destination folder if it doesn't exist
os.makedirs(mod_dst_dir, exist_ok=True)

# Collect .log files in the failed folder
failed_hashes = [f.replace(".log", "") for f in os.listdir(failed_log_dir) if f.endswith(".log")]

# Copy matching .mod files
copied = 0
for hash_id in failed_hashes:
    mod_filename = f"{hash_id}.mod"
    src = os.path.join(mod_src_dir, mod_filename)
    dst = os.path.join(mod_dst_dir, mod_filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        print(f"⚠️ Missing .mod file for: {mod_filename}")

print(f"✅ Copied {copied} .mod files to '{mod_dst_dir}'")


✅ Copied 128 .mod files to '/gpfs/gibbs/project/mcdougal/imc33/mod-extract/data/raw/nmodl-failed'


# Move both failed folders to the code folder

In [10]:
import os
import shutil

# Get current working directory (assuming you're in code/)
code_dir = os.getcwd()

# Define source directories
nmodl_src = os.path.abspath(os.path.join(code_dir, "..", "data", "raw", "nmodl-failed"))
log_src = os.path.abspath(os.path.join(code_dir, "..", "logs", "failed"))

# Define destination directories
failed_root = os.path.join(code_dir, "failed")
nmodl_dst = os.path.join(failed_root, "nmodl")
log_dst = os.path.join(failed_root, "log")

# Create destination folders
os.makedirs(nmodl_dst, exist_ok=True)
os.makedirs(log_dst, exist_ok=True)

# Move all files from nmodl-failed → code/failed/nmodl
for f in os.listdir(nmodl_src):
    src_file = os.path.join(nmodl_src, f)
    dst_file = os.path.join(nmodl_dst, f)
    shutil.move(src_file, dst_file)

# Move all files from logs/failed → code/failed/log
for f in os.listdir(log_src):
    src_file = os.path.join(log_src, f)
    dst_file = os.path.join(log_dst, f)
    shutil.move(src_file, dst_file)

# Optionally remove empty source folders
os.rmdir(nmodl_src)
os.rmdir(log_src)

print(f"✅ Moved nmodl-failed → {nmodl_dst}")
print(f"✅ Moved logs/failed → {log_dst}")

✅ Moved nmodl-failed → /gpfs/gibbs/project/mcdougal/imc33/mod-extract/code/failed/nmodl
✅ Moved logs/failed → /gpfs/gibbs/project/mcdougal/imc33/mod-extract/code/failed/log


In [11]:
!git add .
!git commit -m "added the failed files to the code-folder"
!git push

[main eca58a7] added the failed files to the code-folder
 257 files changed, 20078 insertions(+), 2 deletions(-)
 create mode 100644 code/failed/log/009c5ef10778371f25f95c9ad812c6f628c8f2a7bd7f0638d151bce8c9b4ec06.log
 create mode 100644 code/failed/log/04734e4b840f68bddc93f0c29bc34f11efdda5684cc5f7f53816dcdcd939e97b.log
 create mode 100644 code/failed/log/0c45dc2f28432ff8bfc6cabac9819029179c0bee85e74e3585d0635bb945d3ae.log
 create mode 100644 code/failed/log/0dccbeb565ff1bcc5db7b432326368919c37b65933a4fea149d9abb20a77a2cb.log
 create mode 100644 code/failed/log/10bceb4b8eddc67187dc443dffd5458371258135295d28015925c3b1e77ac1b6.log
 create mode 100644 code/failed/log/11cbdd1f59a9771461f9f08d7c8d2c09ddb7baf1716c09ab354c5f89b06cbcee.log
 create mode 100644 code/failed/log/11de765f9c0362d05675c37bdf1f96987b9593eb91cda851f0108f26389d5151.log
 create mode 100644 code/failed/log/1cb288b6b10521eb86bf8a33c0eedc8ffa962f714c9845465b379389e1841e45.log
 create mode 100644 code/failed/log/1e47dfd343d